# EDA for OilSlick Sentinel-1 GeoTIFF Data

In [ ]:
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio

from IPython.display import display
from scipy.stats import zscore

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'viridis'

## Load metadata

The dataset is situated in `waterbench_data/data/OilSlick`. The metadata table stores relative paths for the GeoTIFFs and the images used.

In [ ]:
BASE_DIR = Path('../10_waterbench_data/data/OilSlick').resolve()
METADATA_PATH = BASE_DIR / 'metadata.csv'

print('Using dataset root:', BASE_DIR)

df = pd.read_csv(METADATA_PATH)
df['label_name'] = df['label'].map({0: 'negative', 1: 'positive'})
df[['sample_id', 'label_name', 'subcategory', 'image_path', 'n_bands']].head()

In [ ]:
print(f'Samples: {len(df):,}')
print('Columns:', ', '.join(df.columns))
print()
display(df[['label_name', 'subcategory','nodata_fraction', 'is_valid', 'n_bands']].value_counts().rename('count').reset_index().head(20))

## GeoTIFF helpers

The GeoTIFFs are multi-band images. The VV and VH bands are the first two bands, and these need to be normalized using z-score to be visualized properly.

In [ ]:
def get_img_path(relative_path: str) -> Path:
    p = Path(relative_path)
    filename = p.name if p.stem.endswith('_s1') else f"{p.stem}_s1{p.suffix}"
    return BASE_DIR / 'images_s1' / filename

def plot_geotiff(sample_row, ax, title, band=1, mean=None, std=None):
    path = get_img_path(sample_row['image_path'])
    
    with rasterio.open(path) as src:
        arr = src.read()
    
    nodata = (arr <= -50) | np.isnan(arr)
    valid = ~nodata.any(axis=0)
    
    image = arr[band].astype(float).copy()
    
    if mean is not None and std is not None:
        image[valid] = (image[valid] - mean[band]) / (std[band] + 1e-6)
    else:
        raise ValueError("Mean and std must be provided for normalization.")
    
    valid_pixels = image[valid]
    p_low = np.percentile(valid_pixels, 2)
    p_high = np.percentile(valid_pixels, 98)
    
    image[valid] = np.clip(image[valid], p_low, p_high)
    
    image_norm = np.full_like(image, np.nan)
    image_norm[valid] = (image[valid] - p_low) / (p_high - p_low + 1e-6)

    ax.imshow(image_norm, cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(title)

## Compute mean / std for normalization

To compute the mean and std, the `random` training split is loaded, and the GeoTIFFs are iterated over (for VV and VH separately) to collect all pixel values. Then, the mean and std are computed across all pixels for each band. 

In [ ]:
"""
For most precision and speed, this RAM-HUNGRY approach
computes the mean and std across all valid pixels in
the training set. A system with 32GB RAM is recommended.
"""

with open(BASE_DIR / 'splits' / 'random' / 'train.txt') as f:
    train_ids = set(line.strip() for line in f)

train_df = df[df['sample_id'].isin(train_ids)]
print(f"Computing mean & std from {len(train_df)} training samples...\n")

all_pixels = {0: [], 1: []}

for idx, (_, row) in enumerate(train_df.iterrows()):
    path = get_img_path(row['image_path'])
    # Skip missing files (for now)
    if path is None or not path.exists():
        continue
    
    with rasterio.open(path) as src:
        arr = src.read()
    
    nodata = (arr <= -50) | np.isnan(arr)
    valid = ~nodata.any(axis=0)
    
    for band in [0, 1]:
        img_band = arr[band].astype(np.float32)
        all_pixels[band].append(torch.from_numpy(img_band[valid].flatten()))
    
    if (idx + 1) % 100 == 0:
        print(f"  Processed: {idx+1}/{len(train_df)}")

means = {}
stds = {}

for band in [0, 1]:
    pixels = torch.cat(all_pixels[band])
    means[band] = pixels.mean().item()
    stds[band] = pixels.std().item()
    print(f"  Band {band}: n={len(pixels):,} pixels, mean={means[band]:.2f} dB, std={stds[band]:.2f} dB")

# (later on, I can provide the final means and stds
# if the step above isn't feasible on machine xyz)

## Metadata Distributions

Analysis of the metadata distributions (labels, subcategories, cloud cover, reflectance, and spatial distribution).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x='label_name', y='reflectance_mean', ax=axes[0])
axes[0].set_title('Reflectance Mean by Class')

sns.histplot(data=df, x='reflectance_mean', bins=30, ax=axes[1])
axes[1].set_title('Reflectance Mean')

plt.tight_layout()

In [ ]:
import folium

subcategory_colors = {
    'Ships': '#FF0000',
    'Platforms': '#FF6B00',
    'Natural seeps': '#00AA00',
    'Negative': '#0066FF'
}

print("Subcategory color mapping:")
print("\033[91m● Ships\033[0m")
print("\033[33m● Platforms\033[0m")
print("\033[92m● Natural seeps\033[0m")
print("\033[94m● Negative\033[0m")
print("The circles are also clickable, showing the sample ID and label details.")

center_lat = df['center_lat'].mean()
center_lon = df['center_lon'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=2)

for _, row in df.iterrows():
    if row['label'] == 1:
        color = subcategory_colors.get(row['subcategory'])
        radius = 4
    else:
        color = subcategory_colors['Negative']
        radius = 3
    
    folium.CircleMarker(
        location=[row['center_lat'], row['center_lon']],
        radius=radius,
        color=color,
        fill=False,
        fillColor=color,
        fillOpacity=0.5,
        weight=1,
        popup=f"{row['sample_id']} ({row['label_name']}: {row['subcategory']})"
    ).add_to(m)

m

# (sometimes, the map below the dots is not rendered?)

## GeoTIFF examples

Positive and negative examples of the original GeoTIFFs are shown below.

In [ ]:
all_samples = df[df['image_path'].apply(lambda img_path: get_img_path(img_path).exists())]

pos_samples = all_samples[all_samples['label'] == 1].iloc[0:2]
neg_samples = all_samples[all_samples['label'] == 0].iloc[0:2]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (_, row) in enumerate(pos_samples.iterrows()):
    plot_geotiff(row, ax=axes[0, i], title=f"Positive VV | {row['sample_id']}", band=0, mean=means, std=stds)

for i, (_, row) in enumerate(neg_samples.iterrows()):
    plot_geotiff(row, ax=axes[0, 2+i], title=f"Negative VV | {row['sample_id']}", band=0, mean=means, std=stds)

for i, (_, row) in enumerate(pos_samples.iterrows()):
    plot_geotiff(row, ax=axes[1, i], title=f"Positive VH | {row['sample_id']}", band=1, mean=means, std=stds)

for i, (_, row) in enumerate(neg_samples.iterrows()):
    plot_geotiff(row, ax=axes[1, 2+i], title=f"Negative VH | {row['sample_id']}", band=1, mean=means, std=stds)

plt.tight_layout()
plt.show()

In [ ]:
all_samples = df[df['image_path'].apply(lambda img_path: get_img_path(img_path).exists())]

pos_samples = all_samples[all_samples['label'] == 1].iloc[6:8]
neg_samples = all_samples[all_samples['label'] == 0].iloc[6:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (_, row) in enumerate(pos_samples.iterrows()):
    plot_geotiff(row, ax=axes[0, i], title=f"Positive VV | {row['sample_id']}", band=0, mean=means, std=stds)

for i, (_, row) in enumerate(neg_samples.iterrows()):
    plot_geotiff(row, ax=axes[0, 2+i], title=f"Negative VV | {row['sample_id']}", band=0, mean=means, std=stds)

for i, (_, row) in enumerate(pos_samples.iterrows()):
    plot_geotiff(row, ax=axes[1, i], title=f"Positive VH | {row['sample_id']}", band=1, mean=means, std=stds)

for i, (_, row) in enumerate(neg_samples.iterrows()):
    plot_geotiff(row, ax=axes[1, 2+i], title=f"Negative VH | {row['sample_id']}", band=1, mean=means, std=stds)

plt.tight_layout()
plt.show()

## Important Note

Now, `pos_00032_s1.tif` is blacked out, since its size is way lower than the usual TIFF size in this dataset.